# Lab 4: Foreign Language to English Translation

**Albert Kabore**  
PhD Student in Artificial Intelligence  
University of the Cumberlands

**Translation direction: English → Romanian.** Parts A and B ask for English to a foreign language, despite the assignment's title. The report follows those instructions and the supplied English-to-Romanian notebook.

**Model:** `google-t5/t5-small`, the checkpoint called `t5-small` in the reference notebook. **Data:** WMT16 `ro-en` test split, 1,999 sentence pairs. **Evaluation:** BLEU, calculated with SacreBLEU.

**Contents:** Introduction · Part A – explanation · Part B – implementation (tasks 1–5) · Part C – analysis · Final Insights · Recommendations · Conclusion · References.

## Introduction

English and Romanian can tell the same news story with different word orders, inflections, and spelling conventions. An automatic comparison with one human translation has to be read with those differences in mind. Here, T5-small translates the 1,999 English sentences in WMT16's Romanian–English test split.

I follow the assignment's chapter 15 notebook, using its model and decoding settings without further training (Surdeanu & Valenzuela-Escárcega, 2024). The report includes the commands and saved results, then examines what the BLEU score reveals and misses. Romanian diacritics and a comparison of greedy decoding with beam search receive particular attention because they affect how these results should be interpreted.

## Part A: Machine Translation (English → Foreign Language)

Machine translation produces text in another language while trying to preserve the source's meaning. For this lab, English sentences pass through a pre-trained T5 model to produce Romanian translations.

Before the model can read a sentence, its tokenizer splits the text into SentencePiece subwords and converts them to integer IDs. A common word may fit into one token; an unfamiliar name or inflection may need several. T5 uses about 32,000 subword pieces, and this checkpoint's tokenizer reports 32,100 entries including special tokens. Character coverage still matters: a letter the tokenizer cannot represent may become an unknown token.

Inside T5, the Transformer encoder uses self-attention to build representations of the English input. The decoder generates Romanian one token at a time, drawing on both the encoded sentence and the tokens already produced. Attention lets it connect related words even when the two languages arrange them differently (Vaswani et al., 2017).

Training this kind of model from scratch would require aligned translations and substantial computation. The existing `google-t5/t5-small` checkpoint already learned English-to-Romanian translation as part of T5's text-to-text training mixture. Adding `translate English to Romanian: ` to an English sentence selects that task (Raffel et al., 2020).

The implementation begins by importing PyTorch, NumPy, and Transformers, setting seeds, and selecting a CPU or GPU. `AutoTokenizer` and `AutoModelForSeq2SeqLM` load the tokenizer and weights. After moving the model to the device, `model.eval()` disables dropout. Hugging Face's `load_dataset` supplies the WMT16 test pairs; only their English text enters the model.

A batch-based `translate` function adds the prefix, tokenizes with padding and truncation, calls `model.generate`, and decodes the output IDs. `Dataset.map` applies it across the test set. Gradient tracking is disabled because inference does not update weights. Greedy decoding picks the most probable next token, while beam search retains several candidates. Either stops at the end-of-sequence token or the output limit.

SacreBLEU compares the predictions with the Romanian references. Its corpus BLEU score combines clipped 1- through 4-gram precisions with a brevity penalty and reports the result on a 0–100 scale (Papineni et al., 2002; Post, 2018). Higher overlap does not necessarily mean better meaning or grammar: valid paraphrases can lose matches, and spelling differences can lower the score. Reading sample translations alongside BLEU helps identify such cases.

## Part B: Model Implementation

The saved run used Python 3.11 with PyTorch, NumPy, Transformers, Datasets, SacreBLEU, SentencePiece, pandas, and Jupyter. Install the packages from the project folder:

```bash
python -m pip install -r requirements.txt
```

For a standalone copy of the notebook, install the same dependencies directly:

```bash
python -m pip install "torch>=2.2" "numpy>=1.26" "transformers>=4.40" "datasets>=2.19" "sacrebleu>=2.4" "sentencepiece>=0.2" "pandas>=2.2" "jupyter>=1.0"
```

Run the cells in order. Greedy decoding took 496 seconds, about eight minutes, on CPU. The optional beam-search run added 4,085 seconds, or about 68 minutes. Hardware and caching affect these timings; a GPU is used when available. Keep `N_SAMPLES = None` to score all 1,999 pairs. A smaller sample is useful for checking the code, but its BLEU score is a different comparison.

### B.1 Initialize the code to enable torch, numpy, and transformers

The first cell imports the libraries, chooses the device, and puts the experiment settings in one place. Seeds support repeatability within a given setup; they cannot guarantee identical results across every device and library release.

In [1]:
import random
import time

import numpy as np
import torch
import transformers
from transformers import set_seed

# Reproducibility: same seed as the reference notebook (seed = 42).
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

# Use a GPU when present; otherwise fall back to CPU.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Run configuration -- identical to the reference notebook (gentlenlp, chapter 15).
MODEL_NAME = "google-t5/t5-small"  # reference uses "t5-small", a Hub redirect to this same checkpoint
SOURCE_LANG, TARGET_LANG = "en", "ro"
TASK_PREFIX = "translate English to Romanian: "
MAX_SOURCE_LENGTH = 1024            # truncation limit for English inputs (tokens)
MAX_TARGET_LENGTH = 128             # max length of generated Romanian output (tokens)
NUM_BEAMS = 1                       # 1 = greedy decoding (reference setting)
BATCH_SIZE = 100
N_SAMPLES = None                    # None = full test set (1,999 pairs); e.g. 200 for a quick run

print(f"torch {torch.__version__} | numpy {np.__version__} | transformers {transformers.__version__}")
print(f"device: {device} | random seed: {SEED}")

torch 2.14.0+cpu | numpy 2.3.4 | transformers 5.18.0
device: cpu | random seed: 42


### B.2 Load tokenizer and pre-trained model

The `Auto` classes select a tokenizer and model from the checkpoint configuration. This run returned `T5Tokenizer` and `T5ForConditionalGeneration`. `model.eval()` switches off dropout. Gradient tracking is handled separately by `torch.inference_mode()` in the translation function (Hugging Face, T5 documentation).

In [2]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"{type(tokenizer).__name__} | vocab size: {tokenizer.vocab_size:,}")
print(f"{type(model).__name__} | parameters: {n_params/1e6:.1f}M")

T5Tokenizer | vocab size: 32,100
T5ForConditionalGeneration | parameters: 60.5M


In [3]:
# Peek at how the tokenizer splits an English sentence into subword pieces.
example = TASK_PREFIX + "The weather is beautiful today."
print(tokenizer.tokenize(example))
print(tokenizer(example).input_ids)

['▁translate', '▁English', '▁to', '▁Romania', 'n', ':', '▁The', '▁weather', '▁is', '▁beautiful', '▁today', '.']
[13959, 1566, 12, 3871, 29, 10, 37, 1969, 19, 786, 469, 5, 1]


### B.3 Load dataset from Hugging Face

The WMT16 news test split, `newstest2016`, contains 1,999 Romanian–English pairs (Bojar et al., 2016). Each row has a `translation` dictionary with `en` and `ro` entries. Although the configuration is named `ro-en`, choosing `en` as the input and `ro` as the reference makes this an English-to-Romanian experiment.

The reference notebook loads `wmt16`; the code below uses its namespaced Hub identifier, `wmt/wmt16`. Romanian references stay outside the model and are used only for scoring. No weights are fitted on these test sentences.

In [4]:
from datasets import load_dataset

# Reference notebook: load_dataset('wmt16', 'ro-en', split='test')  -> same data, namespaced id
dataset = load_dataset("wmt/wmt16", "ro-en", split="test")
if N_SAMPLES is not None:
    dataset = dataset.select(range(N_SAMPLES))

print(dataset)
for row in dataset.select(range(3)):
    print("EN:", row["translation"]["en"])
    print("RO:", row["translation"]["ro"])
    print()

Dataset({
    features: ['translation'],
    num_rows: 1999
})
EN: UN Chief Says There Is No Military Solution in Syria
RO: Șeful ONU declară că nu există soluții militare în Siria

EN: Secretary-General Ban Ki-moon says his response to Russia's stepped up military support for Syria is that "there is no military solution" to the nearly five-year conflict and more weapons will only worsen the violence and misery for millions of people.
RO: Secretarul General Ban Ki-moon afirmă că răspunsul său la suportul militar al Rusiei pentru Siria este că „nu există o soluție militară” la conflictul care durează de aproape cinci ani iar mai multe arme nu ar face decât să agraveze violența și suferința a milioane de oameni.

EN: The U.N. chief again urged all parties, including the divided U.N. Security Council, to unite and support inclusive negotiations to find a political solution.
RO: Șeful ONU a solicitat din nou tuturor părților, inclusiv Consiliului de securitate ONU divizat să se unifice și 

### B.4 Implement the translate method and apply it on the dataset

`translate` accepts a batch of rows, prefixes the English sentences with the T5 task instruction, and produces Romanian text. It returns two lists: `prediction` and the corresponding human `reference`. The following `Dataset.map` call processes the test set in batches of 100.

In [5]:
@torch.inference_mode()  # no gradient tracking -> less memory, faster
def translate(batch, num_beams=NUM_BEAMS):
    """Translate a batch of WMT16 rows from English to Romanian.

    Args:
        batch: dict with key "translation" -> list of {"en": str, "ro": str}.
        num_beams: 1 = greedy decoding; >1 = beam search.
    Returns:
        dict with "reference" (human Romanian) and "prediction" (model Romanian) lists.
    """
    sources = [TASK_PREFIX + pair[SOURCE_LANG] for pair in batch["translation"]]
    targets = [pair[TARGET_LANG] for pair in batch["translation"]]
    inputs = tokenizer(
        sources,
        max_length=MAX_SOURCE_LENGTH,
        truncation=True,       # cut inputs longer than MAX_SOURCE_LENGTH
        padding=True,          # pad to the longest sentence in the batch
        return_tensors="pt",
    ).to(device)
    output_ids = model.generate(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,   # ignore padding positions
        num_beams=num_beams,
        do_sample=False,       # explicitly disable sampling for greedy/beam decoding
        max_length=MAX_TARGET_LENGTH,
    )
    decoded = tokenizer.batch_decode(output_ids, skip_special_tokens=True)
    return {"reference": targets, "prediction": decoded}


# Sanity check on the reference notebook's example sentence.
print(translate({"translation": [{"en": "this is a test", "ro": ""}]})["prediction"][0])

Acesta este un test


In [6]:
start = time.perf_counter()
results = dataset.map(
    translate,
    batched=True,
    batch_size=BATCH_SIZE,
    remove_columns=dataset.column_names,   # keep only reference + prediction
    desc="Translating (greedy)",
)
elapsed = time.perf_counter() - start
print(f"Translated {len(results):,} sentences in {elapsed:.0f}s "
      f"({len(results)/elapsed:.1f} sentences/s on {device})")
results.to_pandas().head(10)

Translated 1,999 sentences in 496s (4.0 sentences/s on cpu)


,reference,prediction
0,Șeful ONU declară că nu există soluții militar...,eful ONU declară că nu există o soluţie milita...
1,Secretarul General Ban Ki-moon afirmă că răspu...,Secretarul General Ban Ki-moon declară că răsp...
2,Șeful ONU a solicitat din nou tuturor părților...,"eful U.N. a cerut din nou tuturor partidelor, ..."
3,Ban a declarat miercuri în cadrul unei conferi...,Ban a declarat la o conferinţă de presă susţin...
4,Ban și-a exprimat regretul că divizările în co...,El şi-a exprimat regretul că diviziunile din c...
5,Ban le-a cerut celor cinci membri permanenți s...,Ban a cerut celor cinci membri permanenţi să d...
6,8 cifre din sondaje care arată că Donald Trump...,8 Sondaje de opinie care arată că Donald Trump...
7,Unii au încercat să îl eticheteze ca politicia...,Unii au încercat să îl eticheteze un eşec.
8,Alții l-au numit o glumă.,Alţii l-au respins ca pe o glumă.
9,Iar alții așteaptă implozia.,Iar unii se opun unei implozii.


In [7]:
# Side-by-side view of source, model output, and human reference for the first 5 test sentences.
for src, row in zip(dataset.select(range(5)), results.select(range(5))):
    print("EN :", src["translation"]["en"])
    print("MT :", row["prediction"])
    print("REF:", row["reference"])
    print("-" * 100)

EN : UN Chief Says There Is No Military Solution in Syria
MT : eful ONU declară că nu există o soluţie militară în Siria
REF: Șeful ONU declară că nu există soluții militare în Siria
----------------------------------------------------------------------------------------------------
EN : Secretary-General Ban Ki-moon says his response to Russia's stepped up military support for Syria is that "there is no military solution" to the nearly five-year conflict and more weapons will only worsen the violence and misery for millions of people.
MT : Secretarul General Ban Ki-moon declară că răspunsul său la sprijinul militar intensificat al Rusiei pentru Siria este că "nu există o soluţie militară" la conflictul de aproape cinci ani şi că alte arme nu vor face altceva decât să înrăutăţească violenţele şi suferinţa a milioane de oameni.
REF: Secretarul General Ban Ki-moon afirmă că răspunsul său la suportul militar al Rusiei pentru Siria este că „nu există o soluție militară” la conflictul care 

### B.5 Evaluate the quality of translations using the BLEU metric

The primary result is corpus BLEU against the unmodified references. In `corpus_score(hypotheses, [references])`, the outer list means there is one complete reference corpus, aligned sentence by sentence with the predictions.

The reference notebook uses `from evaluate import load` and `load("sacrebleu")`. Calling SacreBLEU directly gives the same calculation with matching settings: `13a` tokenization, case-sensitive matching, and exponential smoothing (Post, 2018; SacreBLEU documentation).

Three optional checks follow the raw score. Rescoring after harmonizing Romanian diacritics measures sensitivity to spelling differences; the normalized score is kept separate because the preprocessing has changed. Sentence scores help locate examples for closer reading, although their mean is not corpus BLEU. Finally, a four-beam run and paired bootstrap intervals test whether the extra decoding work brings a clear gain. These checks extend the assignment's five required tasks.

In [8]:
import sacrebleu

hypotheses = results["prediction"]
references = results["reference"]

# 1) Raw corpus BLEU -- greedy decoding, same configuration as the reference notebook
bleu_metric = sacrebleu.BLEU()          # defaults: 13a tokenizer, case-sensitive, exp smoothing
bleu_raw = bleu_metric.corpus_score(hypotheses, [references])
print("Raw BLEU :", bleu_raw)
print("Signature:", bleu_metric.get_signature())
print({"score": bleu_raw.score, "counts": bleu_raw.counts, "totals": bleu_raw.totals,
       "precisions": bleu_raw.precisions, "bp": bleu_raw.bp,
       "sys_len": bleu_raw.sys_len, "ref_len": bleu_raw.ref_len})

# Published result in the reference notebook (same model, data, and decoding; run on a GPU):
REFERENCE_NOTEBOOK_BLEU = 25.18405390123436
print(f"\nReference notebook BLEU: {REFERENCE_NOTEBOOK_BLEU:.2f} | this run: {bleu_raw.score:.2f} "
      f"| difference: {bleu_raw.score - REFERENCE_NOTEBOOK_BLEU:+.2f}")

Raw BLEU : BLEU = 25.18 55.9/31.5/19.2/11.9 (BP = 1.000 ratio = 1.006 hyp_len = 49236 ref_len = 48945)
Signature: nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp|version:2.6.0
{'score': 25.18405390123436, 'counts': [27521, 14902, 8681, 5141], 'totals': [49236, 47237, 45240, 43245], 'precisions': [55.89609229019417, 31.547304020153693, 19.188770999115828, 11.888079546768413], 'bp': 1.0, 'sys_len': 49236, 'ref_len': 48945}

Reference notebook BLEU: 25.18 | this run: 25.18 | difference: +0.00


In [9]:
# Inspect the character mismatch: how often each s/t diacritic form appears in references vs. model
# output, and whether the T5 tokenizer can represent it (encode -> decode round trip without <unk>).
forms = ["ș", "ş", "ț", "ţ", "Ș", "Ş", "Ț", "Ţ"]
print(f"{'char':<6}{'Unicode':<9}{'style':<14}{'in vocab':>9}{'references':>12}{'model output':>14}")
for ch in forms:
    style = "comma-below" if ord(ch) in (0x218, 0x219, 0x21A, 0x21B) else "cedilla"
    in_vocab = tokenizer.decode(tokenizer.encode(ch, add_special_tokens=False)) == ch
    print(f"{ch:<6}U+{ord(ch):04X}   {style:<14}{str(in_vocab):>9}"
          f"{sum(r.count(ch) for r in references):>12,}{sum(h.count(ch) for h in hypotheses):>14,}")
print("\nExample:", tokenizer.tokenize("Șeful ONU"), "->",
      repr(tokenizer.decode(tokenizer.encode("Șeful ONU", add_special_tokens=False))))

char  Unicode  style          in vocab  references  model output


ș     U+0219   comma-below        True       1,882            26


ş     U+015F   cedilla            True           0         1,995
ț     U+021B   comma-below        True       1,753            50


ţ     U+0163   cedilla            True           0         2,379


Ș     U+0218   comma-below       False          40             0
Ş     U+015E   cedilla           False           0             0


Ț     U+021A   comma-below       False           6             0
Ţ     U+0162   cedilla           False           0             0

Example: ['▁', 'Ș', 'e', 'ful', '▁', 'ONU'] -> '<unk>eful ONU'


In [10]:
# 2) Diacritic-normalized BLEU: map comma-below and cedilla variants to the same character.
DIACRITIC_MAP = str.maketrans({"ș": "ş", "Ș": "Ş", "ț": "ţ", "Ț": "Ţ"})

def normalize_ro(text: str) -> str:
    return text.translate(DIACRITIC_MAP)

bleu_norm = sacrebleu.corpus_bleu(
    [normalize_ro(h) for h in hypotheses], [[normalize_ro(r) for r in references]]
)
print("Normalized BLEU:", bleu_norm)

Normalized BLEU: BLEU = 30.23 59.8/36.3/23.9/16.1 (BP = 1.000 ratio = 1.006 hyp_len = 49236 ref_len = 48945)


In [11]:
# 3) Sentence-level BLEU distribution (normalized) and summary table.
sent_bleu = np.array([
    sacrebleu.sentence_bleu(normalize_ro(h), [normalize_ro(r)]).score
    for h, r in zip(hypotheses, references)
])

print(f"{'Metric':<34}{'Raw':>8}{'Normalized':>12}")
print(f"{'Corpus BLEU':<34}{bleu_raw.score:>8.2f}{bleu_norm.score:>12.2f}")
for i, n in enumerate(["1-gram", "2-gram", "3-gram", "4-gram"]):
    print(f"{n + ' precision (%)':<34}{bleu_raw.precisions[i]:>8.1f}{bleu_norm.precisions[i]:>12.1f}")
print(f"{'Brevity penalty':<34}{bleu_raw.bp:>8.3f}{bleu_norm.bp:>12.3f}")
print(f"{'Hyp/ref length ratio':<34}{bleu_raw.sys_len/bleu_raw.ref_len:>8.3f}{bleu_norm.sys_len/bleu_norm.ref_len:>12.3f}")

print("\nSentence-level BLEU (normalized):")
print(f"  mean {sent_bleu.mean():.1f} | median {np.median(sent_bleu):.1f} | "
      f"std {sent_bleu.std():.1f} | share >= 50: {(sent_bleu >= 50).mean():.0%} | "
      f"share < 10: {(sent_bleu < 10).mean():.0%}")

Metric                                 Raw  Normalized
Corpus BLEU                          25.18       30.23
1-gram precision (%)                  55.9        59.8
2-gram precision (%)                  31.5        36.3
3-gram precision (%)                  19.2        23.9
4-gram precision (%)                  11.9        16.1
Brevity penalty                      1.000       1.000
Hyp/ref length ratio                 1.006       1.006

Sentence-level BLEU (normalized):
  mean 27.7 | median 24.4 | std 18.4 | share >= 50: 11% | share < 10: 16%


In [12]:
# Best and worst sentences, to ground the numbers in concrete examples.
order = np.argsort(sent_bleu)
for label, idx in [("BEST", order[-2:][::-1]), ("WORST", order[:2])]:
    for i in idx:
        i = int(i)
        print(f"[{label}] sentence BLEU = {sent_bleu[i]:.1f}")
        print("  EN :", dataset[i]["translation"]["en"])
        print("  MT :", hypotheses[i])
        print("  REF:", references[i])
    print()

[BEST] sentence BLEU = 100.0
  EN : Let us not be hypocrites.
  MT : Să nu fim ipocriţi.
  REF: Să nu fim ipocriți.
[BEST] sentence BLEU = 100.0
  EN : Yes, Chinatown
  MT : Da, Chinatown
  REF: Da, Chinatown

[WORST] sentence BLEU = 0.0
  EN : The great migrations
  MT : Migraţiile mari
  REF: Marile migrații
[WORST] sentence BLEU = 0.0
  EN : Risks
  MT : Riscurile
  REF: Riscuri



In [13]:
# 4) Extension: does beam search beat greedy decoding? Same model, data, and metric; only num_beams changes.
start = time.perf_counter()
results_beam = dataset.map(
    lambda b: translate(b, num_beams=4),
    batched=True, batch_size=BATCH_SIZE,
    remove_columns=dataset.column_names, desc="Translating (beam=4)",
)
beam_secs = time.perf_counter() - start
hyp_beam = results_beam["prediction"]
bleu_beam_raw = sacrebleu.corpus_bleu(hyp_beam, [references])
bleu_beam_norm = sacrebleu.corpus_bleu([normalize_ro(h) for h in hyp_beam],
                                       [[normalize_ro(r) for r in references]])

print(f"{'Decoding':<16}{'Raw BLEU':>10}{'Norm. BLEU':>12}{'BP':>8}{'Time (s)':>10}")
print(f"{'greedy (k=1)':<16}{bleu_raw.score:>10.2f}{bleu_norm.score:>12.2f}{bleu_raw.bp:>8.3f}{elapsed:>10.0f}")
print(f"{'beam (k=4)':<16}{bleu_beam_raw.score:>10.2f}{bleu_beam_norm.score:>12.2f}{bleu_beam_raw.bp:>8.3f}{beam_secs:>10.0f}")


Decoding          Raw BLEU  Norm. BLEU      BP  Time (s)
greedy (k=1)         25.18       30.23   1.000       496
beam (k=4)           25.32       30.37   1.000      4085


In [14]:
# Paired bootstrap resampling (Koehn, 2004): is the beam-vs-greedy BLEU difference significant?
# Corpus BLEU is a function of summed n-gram statistics, so we collect them per sentence once and
# re-aggregate them for each resample of sentence indices.
from sacrebleu.metrics import BLEU

def sentence_stats(hyps, refs):
    rows = []
    for h, r in zip(hyps, refs):
        s = sacrebleu.sentence_bleu(h, [r])
        rows.append(s.counts + s.totals + [s.sys_len, s.ref_len])
    return np.array(rows, dtype=np.int64)            # shape (n, 10)

def bleu_from_stats(stats):
    t = stats.sum(axis=0)
    return BLEU.compute_bleu(correct=t[0:4].tolist(), total=t[4:8].tolist(),
                             sys_len=int(t[8]), ref_len=int(t[9]), smooth_method="exp").score

stats_greedy = sentence_stats(hypotheses, references)
stats_beam = sentence_stats(hyp_beam, references)
# Check: re-aggregated statistics reproduce sacrebleu's corpus scores exactly.
assert abs(bleu_from_stats(stats_greedy) - bleu_raw.score) < 1e-6
assert abs(bleu_from_stats(stats_beam) - bleu_beam_raw.score) < 1e-6

rng = np.random.default_rng(SEED)
n, B = len(references), 1000
diffs, greedy_bs, beam_bs = [], [], []
for _ in range(B):
    idx = rng.integers(0, n, n)
    g, b = bleu_from_stats(stats_greedy[idx]), bleu_from_stats(stats_beam[idx])
    greedy_bs.append(g); beam_bs.append(b); diffs.append(b - g)
diffs = np.array(diffs)
lo, hi = np.percentile(greedy_bs, [2.5, 97.5])
print(f"Greedy BLEU {bleu_raw.score:.2f}  95% CI [{lo:.2f}, {hi:.2f}]")
lo, hi = np.percentile(beam_bs, [2.5, 97.5])
print(f"Beam-4 BLEU {bleu_beam_raw.score:.2f}  95% CI [{lo:.2f}, {hi:.2f}]")
lo, hi = np.percentile(diffs, [2.5, 97.5])
print(f"Beam - greedy: {bleu_beam_raw.score - bleu_raw.score:+.2f}  95% CI [{lo:+.2f}, {hi:+.2f}] | "
      f"share of resamples where beam <= greedy: {(diffs <= 0).mean():.3f}")

Greedy BLEU 25.18  95% CI [24.44, 25.90]
Beam-4 BLEU 25.32  95% CI [24.63, 26.05]
Beam - greedy: +0.14  95% CI [-0.07, +0.35] | share of resamples where beam <= greedy: 0.115


In [15]:
# Save all system outputs next to the source and reference so every number above can be audited.
import pathlib
import pandas as pd

out_dir = pathlib.Path("results"); out_dir.mkdir(exist_ok=True)
table = pd.DataFrame({
    "id": range(len(references)),
    "en": [row["translation"]["en"] for row in dataset],
    "reference_ro": references,
    "greedy_ro": hypotheses,
    "beam4_ro": hyp_beam,
    "greedy_sentence_bleu_norm": sent_bleu.round(2),
})
table.to_csv(out_dir / "translations.csv", index=False, encoding="utf-8")
print(f"saved {len(table):,} rows -> {out_dir / 'translations.csv'}")

saved 1,999 rows -> results\translations.csv


## Part C: Analysis

### C.1 How the commands in Part B work

**B.1 — Setup.** PyTorch executes the model and handles tensors, NumPy supports the numerical analysis, and Transformers supplies the model interfaces. `random.seed`, `np.random.seed`, `torch.manual_seed`, and `set_seed` initialize the random generators used here. Greedy decoding does not sample, but numerical behavior can still differ across hardware and software versions. The `torch.cuda.is_available()` check chooses GPU or CPU; constants hold the model name, language keys, prefix, batch size, sample count, and length limits.

**B.2 — Loading the checkpoint.** The two `from_pretrained` calls retrieve vocabulary rules, configuration, and weights from the Hub or local cache. The saved output identifies a `T5Tokenizer` with 32,100 entries and a `T5ForConditionalGeneration` model with 60.5 million parameters. Its language-model head converts decoder states into vocabulary scores. `.to(device)` moves the weights to the selected hardware, and `model.eval()` disables dropout. The tokenization example splits `Romanian` into subwords and appends end-of-sequence ID `1` (Hugging Face, T5 documentation).

**B.3 — Reading the data.** `load_dataset("wmt/wmt16", "ro-en", split="test")` loads and caches the official test split. `dataset.select` can restrict the run to a smaller sample. Input and reference rows remain aligned throughout; local copies are available in `data/` (Bojar et al., 2016; Hugging Face, Datasets documentation).

**B.4 — Generating a translation.** The function first prepends `translate English to Romanian: ` to each English sentence. Its Romanian counterpart is collected for evaluation and never passed to `generate`.

Calling the tokenizer with `return_tensors="pt"` produces PyTorch input IDs and an attention mask. Padding fits different sentence lengths into one batch, while the mask marks positions the model should ignore. Inputs longer than 1,024 tokens are truncated. Moving these tensors to `device` puts them beside the model weights.

With `num_beams=1` and `do_sample=False`, `model.generate` chooses the most probable token at each step. It stops at the end-of-sequence token or the 128-token length cap. Setting `num_beams=4` retains four candidate sequences under the generation scoring rule. `batch_decode(..., skip_special_tokens=True)` then converts IDs to text and removes special tokens, including padding, end-of-sequence, and unknown tokens.

`@torch.inference_mode()` saves memory by disabling gradient tracking. No parameters are updated. The `map` call applies this function to batches of 100 and replaces the original columns with `reference` and `prediction`. Processing 1,999 sentences in 496 seconds worked out to roughly four sentences per second on the saved run's CPU.

**B.5 — Calculating BLEU.** SacreBLEU tokenizes predictions and references with `13a`. For each n-gram order from one to four, it clips a prediction's matching count to the reference count, then sums counts across sentences. With precisions expressed as fractions, the reported score is **BLEU = 100 × BP × exp(¼ Σ log pₙ)**. For nonempty output, **BP = min(1, exp(1 − r/c))**, where c and r are the system and reference corpus lengths (Papineni et al., 2002).

The printed signature, `nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp|version:2.6.0`, records the reference count, case handling, four n-gram orders, tokenizer, smoothing, and package version. These settings are needed for a fair score comparison (Post, 2018).

`normalize_ro` maps the specified comma-below and cedilla characters to matching forms on both sides. `sentence_bleu` scores individual segments with smoothing and effective n-gram order. For the decoding comparison, the paired bootstrap draws the same sentence indices for both systems 1,000 times and recomputes corpus BLEU from their summed counts. Assertions check that the original counts recover the corpus scores. The fraction of resamples with a nonpositive difference is reported descriptively, rather than as a formal p-value (Koehn, 2004). The last cell saves every source, reference, prediction, and sentence score to `results/translations.csv` using pandas.

### C.2 Explanation of the evaluation results (Part B.5)

The table uses all 1,999 test pairs, with one Romanian reference per sentence.

| Setting | BLEU | 1/2/3/4-gram precision (%) | BP | Length ratio |
|---|---|---|---|---|
| Greedy, raw (primary result) | **25.18** | 55.9 / 31.5 / 19.2 / 11.9 | 1.000 | 1.006 |
| Greedy, diacritic-normalized | **30.23** | 59.8 / 36.3 / 23.9 / 16.1 | 1.000 | 1.006 |
| Beam-4, raw | 25.32 | 56.0 / 31.8 / 19.3 / 11.9 | 1.000 | 1.008 |
| Beam-4, diacritic-normalized | 30.37 | 59.9 / 36.5 / 24.1 / 16.1 | 1.000 | 1.008 |

The raw greedy score is 25.18405390123436, exactly the published reference result. Its n-gram matches, 27,521 / 14,902 / 8,681 / 5,141, and corpus lengths, 49,236 / 48,945, also agree. That establishes the same aggregate evaluation for this saved run. It does not prove that every translation, or every future run, must be identical.

Reading 25.18 as percentage accuracy would be misleading. BLEU counts overlap with the reference under particular scoring settings. Here, modified precision falls from 55.9% for single tokens to 11.9% for four-token sequences. Word choice, order, inflection, spelling, and translation mistakes can all reduce those matches. BP is 1.0 because the system corpus is slightly longer than the reference, with a length ratio of 1.006. Individual sentences can still contain omissions, additions, or truncation.

Spelling accounts for a sizeable part of the score difference. The references contain 1,882 ș and 1,753 ț. Model outputs mostly use their cedilla counterparts, with 1,995 ş and 2,379 ţ, compared with just 26 ș and 50 ț. Mapping the variants to common forms raises BLEU by **5.05 points**, to 30.23. The translations have not changed; only their treatment during scoring has. Raw and normalized scores therefore need separate labels and comparisons under matching preprocessing.

A missing letter is a different problem. The tokenizer's round-trip check cannot preserve uppercase Ș/Ş/Ț/Ţ: `Șeful ONU` becomes `<unk>eful ONU`. Once special tokens are removed, `eful ONU` remains. Normalizing diacritics cannot put that initial letter back. A model/tokenizer with suitable character coverage, or vocabulary expansion followed by embedding updates and training, would be needed. Fine-tuning with the unchanged tokenizer cannot make it emit an unavailable character.

Sentence scores show how much the result varies across the test set. After normalization, their mean is 27.7, median 24.4, and standard deviation 18.4. About 11% score at least 50; 16% fall below 10. These are segment statistics, not an alternative calculation of corpus BLEU. The examples explain why a score alone is insufficient:

- `Să nu fim ipocriţi.` translates “Let us not be hypocrites.” and matches the reference after normalization, scoring 100.
- “The great migrations” becomes `Migraţiile mari`, but the reference says `Marile migrații`. Different inflections and word order leave the core meaning intact while producing a score of zero.
- For “Risks,” the model gives `Riscurile` instead of `Riscuri`, adding the definite article: “the risks.” It also scores zero, despite retaining the main concept.
- The U.N. example contains clearer errors. “Parties” becomes `partidelor`, meaning political parties, where the context calls for `părților`, or sides. The phrase `să unească` lacks the reflexive `se` needed for “to unite.” The output also keeps `U.N.` where the reference uses Romanian `ONU`.

Beam search made little difference on this dataset. Four beams raised raw BLEU from 25.18 to 25.32, a gain of 0.14, but took 4,085 seconds against 496 seconds for greedy decoding: about 8.2 times longer. The paired bootstrap 95% interval for the difference, [−0.07, +0.35], includes zero. In 11.5% of resamples, beam search was no better than greedy. That fraction is not a formal p-value, and the interval does not establish a reliable improvement. This exploratory test gives little reason to spend the extra decoding time here; future settings should be chosen on validation data before the final test.

News sentences with a single reference cannot establish performance across other domains. BLEU also cannot directly judge fluency or meaning, and the input/output caps may affect long sentences. Human review and additional references would help distinguish valid wording differences from errors. The saved CSV makes it possible to inspect all 1,999 outputs rather than rely on the few examples printed above.

## Final Insights

The character checks change where I would put the next round of effort. In `eful ONU`, an unsupported letter leaves a visible defect before questions of word choice or fluency even arise. Increasing the beam width cannot fix missing vocabulary coverage.

The two BLEU scores answer different questions. Raw BLEU preserves the published comparison; the normalized score shows how strongly Romanian spelling variants affect exact matching. Keeping both makes the source of the difference clear. Neither removes the need to read the translations.

## Recommendations

1. Start with the tokenizer's Romanian character coverage. Test uppercase and lowercase diacritics before comparing models, and verify any proposed replacement with actual translations. Vocabulary expansion would also require model updates and training.
2. Keep greedy decoding as the baseline for this dataset. The measured beam-search gain is too uncertain to justify its runtime here. Any further decoding comparison should use validation data and report elapsed time as well as quality.
3. A review of the outputs should include ambiguous words, named entities, grammatical agreement, and long sentences near the length limits. Additional human references, where available, would help check valid alternative wording.
4. For fine-tuning or model selection, use distinct training and validation sets. Leave the test split for the final evaluation, with the raw and normalized scoring procedures stated explicitly.
5. Save the predictions with the commands, dataset identifiers, library versions, and BLEU signature. Runtime comparisons also need hardware and caching details; the 496-second result describes this run's setup.

## Conclusion

T5-small provides a working English-to-Romanian baseline, with raw BLEU of 25.18 on the WMT16 test set. The commands cover the assignment's five tasks, and the saved outputs allow the results to be checked. The main remaining issues are specific: missing Romanian characters, word-sense errors, grammatical mistakes, and a metric sensitive to valid wording differences. Those observations give the next experiment clearer priorities than a small gain from wider beam search.

## References

- Surdeanu, M., & Valenzuela-Escárcega, M. A. (2024). *Deep Learning for Natural Language Processing: A Gentle Introduction*. Cambridge University Press. Chapter 15 reference notebook: https://github.com/clulab/gentlenlp/blob/main/notebooks/chap15_translation_en_to_ro.ipynb
- Raffel, C., Shazeer, N., Roberts, A., Lee, K., Narang, S., Matena, M., Zhou, Y., Li, W., & Liu, P. J. (2020). Exploring the limits of transfer learning with a unified text-to-text transformer. *Journal of Machine Learning Research, 21*(140), 1–67. https://jmlr.org/papers/v21/20-074.html
- Vaswani, A., Shazeer, N., Parmar, N., Uszkoreit, J., Jones, L., Gomez, A. N., Kaiser, Ł., & Polosukhin, I. (2017). Attention is all you need. *Advances in Neural Information Processing Systems, 30*. https://papers.nips.cc/paper/7181-attention-is-all-you-need
- Papineni, K., Roukos, S., Ward, T., & Zhu, W.-J. (2002). BLEU: A method for automatic evaluation of machine translation. *Proceedings of the 40th Annual Meeting of the ACL*, 311–318. https://aclanthology.org/P02-1040/
- Post, M. (2018). A call for clarity in reporting BLEU scores. *Proceedings of the Third Conference on Machine Translation*, 186–191. https://aclanthology.org/W18-6319/
- Koehn, P. (2004). Statistical significance tests for machine translation evaluation. *Proceedings of EMNLP 2004*, 388–395. https://aclanthology.org/W04-3250/
- Bojar, O., et al. (2016). Findings of the 2016 Conference on Machine Translation (WMT16). *Proceedings of the First Conference on Machine Translation*, 131–198. https://aclanthology.org/W16-2301/
- Hugging Face. (n.d.). *T5 documentation*. https://huggingface.co/docs/transformers/model_doc/t5
- Hugging Face. (n.d.). *Datasets: Load*. https://huggingface.co/docs/datasets/loading
- Hugging Face. (n.d.). *WMT16 dataset card*. https://huggingface.co/datasets/wmt/wmt16
- Hugging Face. (n.d.). *T5-small model card*. https://huggingface.co/google-t5/t5-small
- Post, M., et al. (n.d.). *SacreBLEU: Reference BLEU implementation and documentation*. https://github.com/mjpost/sacrebleu